# Codebase

**Objective:** Use one version-controlled codebase for a service and identify every deployment by an exact revision.

## Simple version

Development, staging, and production are deployments of the same codebase—not separate copies of the source.

In [ ]:
# Every environment uses the same repository; only the deployed commit may differ.
codebase = "https://example.com/task-api.git"

deployments = {
    "development": {"codebase": codebase, "commit": "a1b2c3d"},
    "staging": {"codebase": codebase, "commit": "a1b2c3d"},
    "production": {"codebase": codebase, "commit": "91f84aa"},
}

print(deployments)

## Polished version

A small registry rejects deployments from an unrelated repository and keeps the deployed revision explicit.

In [ ]:
from dataclasses import dataclass


# Immutable value objects keep deployment records explicit and safe to share.
@dataclass(frozen=True)
class Codebase:
    service: str
    repository: str


@dataclass(frozen=True)
class Deployment:
    environment: str
    codebase: Codebase
    commit: str


# This registry owns the deployment history for one service.
class DeploymentRegistry:
    def __init__(self, codebase: Codebase) -> None:
        self.codebase = codebase
        self.deployments: dict[str, Deployment] = {}

    def record(self, environment: str, commit: str) -> Deployment:
        if not commit.strip():
            raise ValueError("commit is required")
        deployment = Deployment(environment, self.codebase, commit)
        self.deployments[environment] = deployment
        return deployment


task_api = Codebase(
    service="task-api",
    repository="https://example.com/task-api.git",
)
registry = DeploymentRegistry(task_api)

# Record the exact source revision running in each environment.
registry.record("development", "a1b2c3d")
registry.record("staging", "a1b2c3d")
registry.record("production", "91f84aa")

print(*registry.deployments.values(), sep="\n")

## Applied in this repository

The REST API and LLM API are separate deployable examples. Each project keeps its application, dependency manifest, lockfile, tests, and run commands together.